# Phase 2 — Exploratory Analysis

Aggregate and visualize the cleaned Ames Housing data. SQL queries run against the local SQLite database; charts are produced with seaborn/matplotlib and plotly.

In [ ]:
import sqlite3
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams.update({'figure.dpi': 120, 'axes.titlesize': 13, 'axes.labelsize': 11})

DB_PATH = Path.cwd().parent / 'data' / 'housing.db'
QUERIES_PATH = Path.cwd().parent / 'sql' / 'queries.sql'

conn = sqlite3.connect(DB_PATH)

def q(sql: str) -> pd.DataFrame:
    return pd.read_sql(sql, conn)

df = q('SELECT * FROM clean_housing')
print(f'{len(df):,} rows loaded')

## 1. Price by neighborhood

In [ ]:
nbhd = q("""
    SELECT neighborhood,
           COUNT(*) AS listings,
           ROUND(AVG(sale_price), 0) AS avg_price,
           ROUND(MIN(sale_price), 0) AS min_price,
           ROUND(MAX(sale_price), 0) AS max_price
    FROM clean_housing
    GROUP BY neighborhood
    ORDER BY avg_price DESC
""")
nbhd

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
nbhd_sorted = nbhd.sort_values('avg_price', ascending=True)
bars = ax.barh(nbhd_sorted['neighborhood'], nbhd_sorted['avg_price'] / 1000,
               color='steelblue', alpha=0.85)
ax.set_xlabel('Average sale price ($000s)')
ax.set_title('Average sale price by neighborhood')
plt.tight_layout()
plt.show()

Neighborhood is the single largest determinant of price range in the dataset. `NridgHt`, `NoRidge`, and `StoneBr` average over $300k while `MeadowV` and `BrDale` average under $110k — a roughly 3x spread across the city.

## 2. Price trend by year sold

In [ ]:
trend = q("""
    SELECT yr_sold,
           COUNT(*) AS listings,
           ROUND(AVG(sale_price), 0) AS avg_price,
           ROUND(AVG(sale_price) / AVG(gr_liv_area), 2) AS avg_price_per_sqft
    FROM clean_housing
    GROUP BY yr_sold
    ORDER BY yr_sold
""")
trend

In [ ]:
fig, ax1 = plt.subplots(figsize=(8, 4))
ax2 = ax1.twinx()
ax1.bar(trend['yr_sold'], trend['listings'], color='steelblue', alpha=0.5, label='Listings')
ax2.plot(trend['yr_sold'], trend['avg_price'] / 1000, color='firebrick',
         marker='o', linewidth=2, label='Avg price ($000s)')
ax1.set_xlabel('Year sold')
ax1.set_ylabel('Listing volume', color='steelblue')
ax2.set_ylabel('Avg sale price ($000s)', color='firebrick')
ax1.set_title('Sale volume and average price by year')
fig.legend(loc='upper right', bbox_to_anchor=(0.88, 0.88))
plt.tight_layout()
plt.show()

Volume dropped sharply in 2010 (reflecting the aftermath of the 2008 financial crisis), but average sale price remained relatively stable across 2006–2010. This suggests the Ames market was more insulated from the national price collapse than major metros.

## 3. Overall quality vs. price

In [ ]:
qual = q("""
    SELECT overall_qual,
           COUNT(*) AS listings,
           ROUND(AVG(sale_price), 0) AS avg_price
    FROM clean_housing
    GROUP BY overall_qual
    ORDER BY overall_qual
""")

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(qual['overall_qual'], qual['avg_price'] / 1000, color='steelblue', alpha=0.85)
ax.set_xlabel('Overall quality rating (1–10)')
ax.set_ylabel('Average sale price ($000s)')
ax.set_title('Average price by overall quality rating')
ax.set_xticks(qual['overall_qual'])
plt.tight_layout()
plt.show()

The relationship between overall quality and price is strongly monotonic — each quality point corresponds to roughly $40–50k in average price up to grade 8, then accelerates sharply for grades 9 and 10. Quality rating will be the dominant predictor in the model.

## 4. Seasonal patterns (month sold)

In [ ]:
seasonal = q("""
    SELECT mo_sold,
           COUNT(*) AS listings,
           ROUND(AVG(sale_price), 0) AS avg_price
    FROM clean_housing
    GROUP BY mo_sold
    ORDER BY mo_sold
""")

month_labels = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

fig, ax1 = plt.subplots(figsize=(9, 4))
ax2 = ax1.twinx()
ax1.bar(seasonal['mo_sold'], seasonal['listings'], color='steelblue', alpha=0.5, label='Listings')
ax2.plot(seasonal['mo_sold'], seasonal['avg_price'] / 1000, color='firebrick',
         marker='o', linewidth=2, label='Avg price ($000s)')
ax1.set_xticks(range(1, 13))
ax1.set_xticklabels(month_labels)
ax1.set_ylabel('Listing volume', color='steelblue')
ax2.set_ylabel('Avg price ($000s)', color='firebrick')
ax1.set_title('Sale volume and average price by month')
fig.legend(loc='upper left', bbox_to_anchor=(0.1, 0.9))
plt.tight_layout()
plt.show()

Sales volume peaks in May–June, consistent with the typical spring buying season, and drops sharply through the winter months. Average prices show less seasonality, suggesting that what sells in winter is not systematically cheaper — just fewer transactions.

## 5. Correlation heatmap — numeric features vs. price

In [ ]:
num_cols = [
    'sale_price', 'overall_qual', 'gr_liv_area', 'total_bsmt_sf',
    'garage_area', 'total_bathrooms', 'age_at_sale', 'lot_area',
    'fireplaces', 'tot_rms_abvgrd', 'wood_deck_sf'
]
corr = df[num_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
    center=0, vmin=-1, vmax=1, linewidths=0.5, ax=ax
)
ax.set_title('Correlation matrix — key features')
plt.tight_layout()
plt.show()

`overall_qual` (0.80), `gr_liv_area` (0.71), and `total_bsmt_sf` (0.64) have the strongest linear correlations with sale price. `age_at_sale` is negatively correlated (-0.56), confirming older homes sell for less even after controlling for remodeling. `lot_area` has a surprisingly weak correlation (0.26), indicating that lot size matters less than interior quality in this market.

## 6. Price distribution by building type (Plotly interactive)

In [ ]:
fig = px.box(
    df,
    x='bldg_type',
    y='sale_price',
    color='bldg_type',
    labels={'bldg_type': 'Building type', 'sale_price': 'Sale price ($)'},
    title='Sale price distribution by building type',
    color_discrete_sequence=px.colors.qualitative.Set2
)
fig.update_layout(showlegend=False, plot_bgcolor='white', paper_bgcolor='white')
fig.show()

Single-family homes (`1Fam`) have the widest price range and the highest median. Two-family dwellings and townhouse end units cluster notably lower. The broad spread in single-family homes reflects the dominance of neighborhood and quality — a topic the model will quantify.

## 7. Price per square foot by decade built

In [ ]:
decade = q("""
    SELECT (year_built / 10) * 10 AS decade_built,
           COUNT(*) AS listings,
           ROUND(AVG(sale_price), 0) AS avg_price,
           ROUND(AVG(CAST(sale_price AS REAL) / gr_liv_area), 2) AS avg_price_per_sqft
    FROM clean_housing
    GROUP BY decade_built
    ORDER BY decade_built
""")

fig = px.bar(
    decade,
    x='decade_built',
    y='avg_price_per_sqft',
    text='listings',
    labels={'decade_built': 'Decade built', 'avg_price_per_sqft': 'Avg price per sq ft ($)'},
    title='Average price per square foot by construction decade',
    color='avg_price_per_sqft',
    color_continuous_scale='Blues'
)
fig.update_traces(textposition='outside')
fig.update_layout(coloraxis_showscale=False, plot_bgcolor='white', paper_bgcolor='white')
fig.show()

Homes built in the 2000s command the highest price per square foot, consistent with buyers paying a premium for modern construction standards. Pre-1940 homes have the lowest price-per-sqft despite sometimes being larger, suggesting maintenance costs and outdated layouts offset their size advantage.

In [ ]:
conn.close()